In [1]:
from google import genai
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [3]:
sample_output = """
Issue: received the wrong size for order #12345
Resolution: Process the exchange request pending policy verification.
Reason: The customer received an incorrect size and requested an exchange.
"""

print(sample_output)


Issue: received the wrong size for order #12345
Resolution: Process the exchange request pending policy verification.
Reason: The customer received an incorrect size and requested an exchange.



In [4]:
required_fields = [
    "Issue",
    "Resolution",
    "Reason"
]

In [5]:
def validate_resolution_output(result):

    lines = result.splitlines()

    extracted_fields = {}

    for line in lines:

        for field in required_fields:

            if line.startswith(field + ":"):

                value = line.split(":", 1)[1].strip()

                extracted_fields[field] = value

    missing_fields = [
        field
        for field in required_fields
        if field not in extracted_fields
    ]

    empty_fields = [
        field
        for field, value in extracted_fields.items()
        if not value
    ]

    return {
        "extracted_fields": extracted_fields,
        "missing_fields": missing_fields,
        "empty_fields": empty_fields,
        "valid": len(missing_fields) == 0 and len(empty_fields) == 0
    }

In [6]:
validation = validate_resolution_output(sample_output)

print(validation)

{'extracted_fields': {'Issue': 'received the wrong size for order #12345', 'Resolution': 'Process the exchange request pending policy verification.', 'Reason': 'The customer received an incorrect size and requested an exchange.'}, 'missing_fields': [], 'empty_fields': [], 'valid': True}


In [7]:
def check_resolution_content(result):

    result_lower = result.lower()

    invalid_phrases = [
        "i don't know",
        "i cannot help",
        "not sure",
        "unable to determine"
    ]

    detected_issues = []

    for phrase in invalid_phrases:

        if phrase in result_lower:
            detected_issues.append(phrase)

    return {
        "content_valid": len(detected_issues) == 0,
        "detected_issues": detected_issues
    }

In [8]:
def check_resolution_output(result):

    structure_check = validate_resolution_output(result)

    content_check = check_resolution_content(result)

    final_valid = (
        structure_check["valid"]
        and content_check["content_valid"]
    )

    return {
        "structure_check": structure_check,
        "content_check": content_check,
        "final_valid": final_valid
    }

In [9]:
result = check_resolution_output(sample_output)

print(result)

{'structure_check': {'extracted_fields': {'Issue': 'received the wrong size for order #12345', 'Resolution': 'Process the exchange request pending policy verification.', 'Reason': 'The customer received an incorrect size and requested an exchange.'}, 'missing_fields': [], 'empty_fields': [], 'valid': True}, 'content_check': {'content_valid': True, 'detected_issues': []}, 'final_valid': True}


In [13]:
def check_output(complaint, resolution):

    prompt = f"""
    You are an output quality checker for an e-commerce
    customer complaint resolution system.

    Customer Complaint:
    {complaint}

    Generated Resolution:
    {resolution}

    Check whether the resolution:

    1. Addresses the customer's actual complaint.
    2. Uses only information available from the complaint.
    3. Avoids inventing order details.
    4. Avoids making unsupported promises.
    5. Provides a reasonable next step.

    Return exactly:

    Status: <PASS or FAIL>
    Reason: <brief reason>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [14]:
complaint = """
I ordered a black Nike shoe with order ID #12345,
but I received the wrong size. I want to exchange it.
"""

resolution = """
Issue: received the wrong size for order #12345
Resolution: Process the exchange request pending policy verification.
Reason: The customer received an incorrect size and requested an exchange.
"""

check_result = check_output(
    complaint,
    resolution
)

print(check_result)

Status: PASS
Reason: The resolution accurately addresses the wrong size complaint for order #12345, relies solely on the provided text without inventing details, avoids unsupported promises by making the exchange pending policy verification, and provides a reasonable next step.
